In [16]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import fagtb_core
importlib.reload(fagtb_core)

from fagtb_core import FAGTB


# ============================================================
# Configurações
# ============================================================

dataset_path = Path("datasets/tratado/hmda.csv")

target_col = "Favorable Outcome (Does Not Reincid)"
sensitive_col = "African-American"

test_size = 0.30
random_state = 42

lambda_fagtb = 0.05


# ============================================================
# Carregamento
# ============================================================

df = pd.read_csv(dataset_path)

y = df[target_col].astype(int)
sensitive = df[sensitive_col].astype(int)

X = df.drop(columns=[target_col])


# ============================================================
# Split
# ============================================================

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X,
    y,
    sensitive,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)


# ============================================================
# Normalização
# ============================================================

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)


# ============================================================
# FAGTB
# ============================================================

modelo_fagtb = FAGTB(
    n_estimators=200,
    learning_rate=0.01,
    min_samples_split=2,
    min_impurity=1e-7,
    max_depth=9,
    max_features=None,
    regression=False
)

modelo_fagtb.fit(
    X_train_scaled,
    y_train.values,
    s_train.values,
    LAMBDA=lambda_fagtb,
    Xtest=X_test_scaled,
    yt=y_test.values,
    sensitivet=s_test.values
)


# ============================================================
# Predição
# ============================================================

y_score = modelo_fagtb.predict(X_test_scaled)
y_pred = (y_score >= 0.5).astype(int)


# ============================================================
# Métricas
# ============================================================

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n==============================")
print("Resultados FAGTB")
print("==============================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

MemoryError: Unable to allocate 58.3 MiB for an array with shape (22, 347177) and data type int64

In [ ]:
def explain_model(
    model,
    X,
    max_display=10,
    nome=None,
    target_name=None,
    cmap="coolwarm",
    max_tempo_min=15
):
    import time
    import shap
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

    # ============================================================
    # Cópia dos dados
    # ============================================================

    X_exp = X.copy()

    # ============================================================
    # Modelos suportados diretamente pelo TreeExplainer
    # ============================================================

    modelos_problema = isinstance(model, DecisionTreeClassifier)

    modelos_sem_problema = isinstance(model, (
        GradientBoostingClassifier,
        DecisionTreeRegressor,
        GradientBoostingRegressor
    ))

    modelos_tree_explainer = isinstance(model, (
        GradientBoostingClassifier,
        DecisionTreeRegressor,
        GradientBoostingRegressor,
        DecisionTreeClassifier
    ))

    figuras = {}

    # ============================================================
    # Caminho rápido: TreeExplainer
    # ============================================================

    if modelos_tree_explainer:

        print("🌳 Usando TreeExplainer")

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_exp)
        X_plot = X_exp

    # ============================================================
    # Caminho genérico: modelo caixa-preta
    # Ex.: FAGTB
    # ============================================================

    else:

        print("⚠️ Tipo de modelo não suportado pelo TreeExplainer.")
        print("🔁 Usando explicabilidade model-agnostic caixa-preta.")

        # ========================================================
        # Garante dados numéricos
        # ========================================================

        X_numeric = X_exp.copy()

        for col in X_numeric.columns:
            X_numeric[col] = pd.to_numeric(
                X_numeric[col],
                errors="coerce"
            )

        X_numeric = X_numeric.fillna(0.0).astype(float)

        n_features = X_numeric.shape[1]

        # ========================================================
        # Controle automático de custo
        # ========================================================

        if n_features > 100:
            background_size = 10
            explain_size = 30

        elif n_features > 60:
            background_size = 20
            explain_size = 50

        elif n_features > 30:
            background_size = 30
            explain_size = 100

        else:
            background_size = 50
            explain_size = 150

        X_background = shap.sample(
            X_numeric,
            min(background_size, len(X_numeric)),
            random_state=42
        )

        X_plot = shap.sample(
            X_numeric,
            min(explain_size, len(X_numeric)),
            random_state=42
        )

        # ========================================================
        # Função de predição explicada pelo SHAP
        # ========================================================

        def predict_score(X_input):

            X_input = pd.DataFrame(
                X_input,
                columns=X_numeric.columns
            )

            if hasattr(model, "predict_proba"):
                return model.predict_proba(X_input)[:, 1]

            return np.asarray(model.predict(X_input))

        # ========================================================
        # Estimador de tempo
        # ========================================================

        print("\n⏱️ Estimando tempo do SHAP caixa-preta...")

        n_teste = min(5, len(X_plot))

        X_teste = X_plot.iloc[:n_teste].copy()

        explainer_teste = shap.Explainer(
            predict_score,
            X_background
        )

        max_evals = 2 * X_plot.shape[1] + 1

        inicio = time.time()

        _ = explainer_teste(
            X_teste,
            max_evals=max_evals,
            silent=True
        )

        tempo_teste = time.time() - inicio

        tempo_por_amostra = tempo_teste / n_teste
        tempo_estimado = tempo_por_amostra * len(X_plot)
        tempo_estimado_min = tempo_estimado / 60

        print(
            f"Tempo teste: {tempo_teste:.2f} s "
            f"para {n_teste} amostras"
        )

        print(
            f"Tempo estimado total: "
            f"{tempo_estimado_min:.2f} min "
            f"para {len(X_plot)} amostras"
        )

        print(
            f"Configuração SHAP caixa-preta: "
            f"{len(X_plot)} amostras explicadas, "
            f"{len(X_background)} background, "
            f"{X_plot.shape[1]} variáveis, "
            f"max_evals={max_evals}"
        )

        # ========================================================
        # Redução automática caso a estimativa seja muito alta
        # ========================================================

        if tempo_estimado_min > max_tempo_min:

            fator = max_tempo_min / tempo_estimado_min

            novo_explain_size = max(
                10,
                int(len(X_plot) * fator)
            )

            print(
                f"\n⚠️ Tempo estimado acima de {max_tempo_min} min."
            )

            print(
                f"Reduzindo amostras explicadas de "
                f"{len(X_plot)} para {novo_explain_size}."
            )

            X_plot = shap.sample(
                X_numeric,
                min(novo_explain_size, len(X_numeric)),
                random_state=42
            )

            tempo_estimado = tempo_por_amostra * len(X_plot)
            tempo_estimado_min = tempo_estimado / 60

            print(
                f"Nova estimativa: {tempo_estimado_min:.2f} min"
            )

        # ========================================================
        # SHAP definitivo
        # ========================================================

        print("\n🚀 Rodando SHAP caixa-preta definitivo...")

        explainer = shap.Explainer(
            predict_score,
            X_background
        )

        inicio = time.time()

        shap_exp = explainer(
            X_plot,
            max_evals=max_evals,
            silent=True
        )

        tempo_real = time.time() - inicio

        print(
            f"✅ SHAP caixa-preta concluído em "
            f"{tempo_real / 60:.2f} min"
        )

        shap_values = shap_exp.values

        if shap_values.shape != X_plot.shape:
            raise AssertionError(
                f"shap_values.shape = {shap_values.shape}, "
                f"X_plot.shape = {X_plot.shape}"
            )

        modelos_problema = False
        modelos_sem_problema = True

    # ============================================================
    # Caso padrão: modelos sem problema
    # ============================================================

    if not modelos_problema:

        if not modelos_sem_problema:
            print("⚠️ Tipo de modelo não identificado, tentando procedimento padrão")

            if shap_values.shape != X_plot.shape:
                raise AssertionError(
                    f"shap_values.shape = {shap_values.shape}, "
                    f"X.shape = {X_plot.shape}"
                )

        print("🔍 Gerando explicabilidade global do modelo...")

        plt.figure()

        shap.summary_plot(
            shap_values,
            X_plot,
            plot_type="bar",
            max_display=max_display,
            show=False
        )

        if nome:
            plt.title(f"{nome} — Importância global")

        figuras["bar"] = plt.gcf()

        plt.figure()

        shap.summary_plot(
            shap_values,
            X_plot,
            max_display=max_display,
            cmap=plt.get_cmap(cmap),
            show=False
        )

        if nome:
            plt.title(f"{nome} — Distribuição dos impactos")

        figuras["beeswarm"] = plt.gcf()

        return figuras

    # ============================================================
    # Caso especial: DecisionTreeClassifier
    # ============================================================

    elif isinstance(model, DecisionTreeClassifier):

        if isinstance(shap_values, list):

            n_classes = len(shap_values)

            for i in range(n_classes):

                print(f"\n📊 {target_name}: {i} — Explicação global")

                plt.figure()

                shap.summary_plot(
                    shap_values[i],
                    X_plot,
                    plot_type="bar",
                    max_display=max_display,
                    show=False
                )

                if nome:
                    plt.title(
                        f"{nome} — {target_name}: {i} — Importância global"
                    )

                figuras[f"classe_{i}_bar"] = plt.gcf()

                print(f"📈 {target_name}: {i} — Distribuição dos impactos")

                plt.figure()

                shap.summary_plot(
                    shap_values[i],
                    X_plot,
                    max_display=max_display,
                    cmap=plt.get_cmap(cmap),
                    show=False
                )

                if nome:
                    plt.title(
                        f"{nome} — {target_name}: {i} — Distribuição dos impactos"
                    )

                figuras[f"classe_{i}_beeswarm"] = plt.gcf()

            return figuras

        else:

            _, _, n_classes = shap_values.shape

            for i in range(n_classes):

                print(f"\n📊 {target_name}: {i} — Explicação global")

                plt.figure()

                shap.summary_plot(
                    shap_values[:, :, i],
                    X_plot,
                    plot_type="bar",
                    max_display=max_display,
                    show=False
                )

                if nome:
                    plt.title(
                        f"{nome} — {target_name}: {i} — Importância global"
                    )

                figuras[f"classe_{i}_bar"] = plt.gcf()

                print(f"📈 {target_name}: {i} — Distribuição dos impactos")

                plt.figure()

                shap.summary_plot(
                    shap_values[:, :, i],
                    X_plot,
                    max_display=max_display,
                    cmap=plt.get_cmap(cmap),
                    show=False
                )

                if nome:
                    plt.title(
                        f"{nome} — {target_name}: {i} — Distribuição dos impactos"
                    )

                figuras[f"classe_{i}_beeswarm"] = plt.gcf()

            return figuras

In [ ]:
import time
import joblib
import shap
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# Pastas
# ============================================================

pasta_resultados = Path("resultados")
pasta_modelos = pasta_resultados / "modelos"
pasta_splits = pasta_resultados / "splits"


# ============================================================
# Função para estimar tempo do SHAP caixa-preta
# ============================================================

def estimar_tempo_shap_caixa_preta(
    model,
    X,
    background_size=20,
    explain_size=50,
    n_teste=5,
    max_evals=None
):
    # ========================================================
    # Garante dados numéricos
    # ========================================================

    X_numeric = X.copy()

    for col in X_numeric.columns:
        X_numeric[col] = pd.to_numeric(
            X_numeric[col],
            errors="coerce"
        )

    X_numeric = X_numeric.fillna(0.0).astype(float)

    # ========================================================
    # Amostras
    # ========================================================

    X_background = shap.sample(
        X_numeric,
        min(background_size, len(X_numeric)),
        random_state=42
    )

    X_plot = shap.sample(
        X_numeric,
        min(explain_size, len(X_numeric)),
        random_state=42
    )

    X_teste = X_plot.iloc[:min(n_teste, len(X_plot))].copy()

    # ========================================================
    # max_evals padrão do PermutationExplainer
    # ========================================================

    if max_evals is None:
        max_evals = 2 * X_numeric.shape[1] + 1

    # ========================================================
    # Função explicada
    # ========================================================

    def predict_score(X_input):

        X_input = pd.DataFrame(
            X_input,
            columns=X_numeric.columns
        )

        if hasattr(model, "predict_proba"):
            return model.predict_proba(X_input)[:, 1]

        return np.asarray(model.predict(X_input))

    # ========================================================
    # Estimativa
    # ========================================================

    explainer = shap.Explainer(
        predict_score,
        X_background
    )

    print("\n============================================")
    print("Estimando SHAP caixa-preta")
    print("============================================")
    print(f"Features: {X_numeric.shape[1]}")
    print(f"Background size: {len(X_background)}")
    print(f"Amostras para explicar: {len(X_plot)}")
    print(f"Amostras no teste: {len(X_teste)}")
    print(f"max_evals: {max_evals}")

    inicio = time.time()

    _ = explainer(
        X_teste,
        max_evals=max_evals,
        silent=True
    )

    tempo_teste = time.time() - inicio

    tempo_por_amostra = tempo_teste / len(X_teste)
    tempo_total = tempo_por_amostra * len(X_plot)

    print("--------------------------------------------")
    print(f"Tempo teste: {tempo_teste:.2f} s")
    print(f"Tempo por amostra: {tempo_por_amostra:.2f} s")
    print(f"Tempo estimado total: {tempo_total:.2f} s")
    print(f"Tempo estimado total: {tempo_total / 60:.2f} min")

    return {
        "n_features": X_numeric.shape[1],
        "background_size": len(X_background),
        "explain_size": len(X_plot),
        "n_teste": len(X_teste),
        "max_evals": max_evals,
        "tempo_teste_s": tempo_teste,
        "tempo_por_amostra_s": tempo_por_amostra,
        "tempo_total_estimado_s": tempo_total,
        "tempo_total_estimado_min": tempo_total / 60
    }

In [ ]:
import time
import joblib
import shap
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression


# ============================================================
# Pastas
# ============================================================

pasta_resultados = Path("resultados")
pasta_modelos = pasta_resultados / "modelos"
pasta_splits = pasta_resultados / "splits"

pasta_resultados.mkdir(parents=True, exist_ok=True)


# ============================================================
# Configurações do teste
# ============================================================

background_size = 20

# Quantidades pequenas só para medir tendência
explain_sizes_teste = [2, 5, 10]

# Tamanhos que você quer estimar sem rodar de verdade
explain_sizes_estimados = [30, 50, 100, 200, 500]


# ============================================================
# Função auxiliar
# ============================================================

def preparar_X_numeric(X):

    X_numeric = X.copy()

    for col in X_numeric.columns:
        X_numeric[col] = pd.to_numeric(
            X_numeric[col],
            errors="coerce"
        )

    X_numeric = X_numeric.fillna(0.0).astype(float)

    return X_numeric


def medir_tempo_shap_fagtb(
    model,
    X,
    background_size=20,
    explain_size=5
):

    X_numeric = preparar_X_numeric(X)

    X_background = shap.sample(
        X_numeric,
        min(background_size, len(X_numeric)),
        random_state=42
    )

    X_plot = shap.sample(
        X_numeric,
        min(explain_size, len(X_numeric)),
        random_state=42
    )

    max_evals = 2 * X_numeric.shape[1] + 1

    def predict_score(X_input):

        X_input = pd.DataFrame(
            X_input,
            columns=X_numeric.columns
        )

        return model.predict_proba(X_input)[:, 1]

    explainer = shap.Explainer(
        predict_score,
        X_background
    )

    inicio = time.time()

    _ = explainer(
        X_plot,
        max_evals=max_evals,
        silent=True
    )

    tempo_s = time.time() - inicio

    return {
        "n_features": X_numeric.shape[1],
        "background_size": len(X_background),
        "explain_size": len(X_plot),
        "max_evals": max_evals,
        "tempo_s": tempo_s,
        "tempo_min": tempo_s / 60,
        "tempo_por_amostra_s": tempo_s / len(X_plot)
    }


# ============================================================
# Testa todos os modelos FAGTB
# ============================================================

resultados = []

modelos_fagtb = sorted(pasta_modelos.glob("*fagtb*.joblib"))

print(f"Modelos FAGTB encontrados: {len(modelos_fagtb)}")

for caminho_modelo in modelos_fagtb:

    nome_modelo = caminho_modelo.stem.lower()
    nome_dataset = nome_modelo.split("_fagtb_")[0]

    print("\n================================================")
    print(f"Modelo:  {nome_modelo}")
    print(f"Dataset: {nome_dataset}")
    print("================================================")

    modelo = joblib.load(caminho_modelo)

    X_test = pd.read_csv(
        pasta_splits / f"{nome_dataset}_X_test.csv"
    )

    for explain_size in explain_sizes_teste:

        print(
            f"\nRodando teste: "
            f"background={background_size}, "
            f"explain_size={explain_size}"
        )

        res = medir_tempo_shap_fagtb(
            model=modelo,
            X=X_test,
            background_size=background_size,
            explain_size=explain_size
        )

        res["modelo"] = nome_modelo
        res["dataset"] = nome_dataset

        resultados.append(res)

        print(
            f"Tempo: {res['tempo_min']:.2f} min | "
            f"{res['tempo_por_amostra_s']:.2f} s/amostra | "
            f"features={res['n_features']}"
        )


df_tempos = pd.DataFrame(resultados)

df_tempos.to_csv(
    pasta_resultados / "tempos_shap_fagtb_medidos.csv",
    index=False
)

df_tempos

In [ ]:
estimativas = []

for nome_modelo, df_modelo in df_tempos.groupby("modelo"):

    X_reg = df_modelo[["explain_size"]].values
    y_reg = df_modelo["tempo_min"].values

    reg = LinearRegression()
    reg.fit(X_reg, y_reg)

    a = reg.coef_[0]
    b = reg.intercept_

    dataset = df_modelo["dataset"].iloc[0]
    n_features = df_modelo["n_features"].iloc[0]
    max_evals = df_modelo["max_evals"].iloc[0]

    for explain_size in explain_sizes_estimados:

        tempo_estimado_min = reg.predict(
            np.array([[explain_size]])
        )[0]

        estimativas.append({
            "modelo": nome_modelo,
            "dataset": dataset,
            "n_features": n_features,
            "max_evals": max_evals,
            "background_size": background_size,
            "explain_size_estimado": explain_size,
            "tempo_estimado_min": tempo_estimado_min,
            "tempo_estimado_horas": tempo_estimado_min / 60,
            "coef_min_por_amostra": a,
            "intercept_min": b
        })

df_estimativas = pd.DataFrame(estimativas)

df_estimativas.to_csv(
    pasta_resultados / "tempos_shap_fagtb_estimados.csv",
    index=False
)

df_estimativas

In [ ]:
df_estimativas.pivot_table(
    index=["dataset", "modelo"],
    columns="explain_size_estimado",
    values="tempo_estimado_min"
).round(2)

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Pastas
# ============================================================

pasta_resultados = Path("resultados")
pasta_splits = pasta_resultados / "splits"

# ============================================================
# Lê tempos medidos
# ============================================================

df_tempos = pd.read_csv(
    pasta_resultados / "tempos_shap_fagtb_medidos.csv"
)

# ============================================================
# Usa apenas pontos mais estáveis
# Evita explain_size=2, que pode conter aquecimento/outlier
# ============================================================

df_base = df_tempos[
    df_tempos["explain_size"] >= 5
].copy()

# ============================================================
# Estima tempo por modelo usando média s/amostra
# ============================================================

estimativas = []

for modelo, df_modelo in df_base.groupby("modelo"):

    dataset = df_modelo["dataset"].iloc[0]

    caminho_X_test = pasta_splits / f"{dataset}_X_test.csv"

    X_test = pd.read_csv(caminho_X_test)

    n_amostras_total = len(X_test)

    tempo_por_amostra_s = df_modelo["tempo_por_amostra_s"].mean()

    tempo_total_s = tempo_por_amostra_s * n_amostras_total

    estimativas.append({
        "modelo": modelo,
        "dataset": dataset,
        "n_features": df_modelo["n_features"].iloc[0],
        "background_size": df_modelo["background_size"].iloc[0],
        "max_evals": df_modelo["max_evals"].iloc[0],
        "n_amostras_X_test": n_amostras_total,
        "tempo_por_amostra_s": tempo_por_amostra_s,
        "tempo_total_estimado_min": tempo_total_s / 60,
        "tempo_total_estimado_horas": tempo_total_s / 3600,
    })

df_estimativa_full = pd.DataFrame(estimativas)

df_estimativa_full.to_csv(
    pasta_resultados / "estimativa_shap_fagtb_Xtest_inteiro.csv",
    index=False
)

df_estimativa_full